##### **ARXIU DE CREACIÓ I VALIDACIÓ DE DADES (CRUD)**

In [1]:
import sys

sys.path.append("../src/")

##### **IMPORTACIÓ DE MÒDULS I MODELS DEL DOMINI**

In [2]:
from sqlalchemy import select, and_, or_, func
from domain import(
    Base,
    engine,
    Session,
    User,
    FinancialProfile,
    Budget,
    Category, 
    BudgetCategory,
    PaymentMethod,
    TransactionType,
    Transaction,
    Tag
    
)
from datetime import time

In [3]:
engine.url

sqlite:///walletly.db

##### **INSERCIÓ D'USUARI**

In [4]:
from sqlalchemy import select

with Session() as session:
    # 1. Busquem si l'usuari ja existeix pel seu email (que és UNIQUE)
    stmt = select(User).where(User.email == "tonimolist@gmail.com")
    usuari_existent = session.scalar(stmt)
    
    if usuari_existent:
        print(f"L'usuari ja existeix: {usuari_existent.name} {usuari_existent.surname}")
    else:
        # 2. Si no existeix, el creem fent servir la MATEIXA sessió
        Toni_Molist = User()
        Toni_Molist.name = "Toni"
        Toni_Molist.surname = "Molist"
        Toni_Molist.email = "tonimolist@gmail.com"
        Toni_Molist.password = "password123"
        
        session.add(Toni_Molist)
        session.commit()
        print("Usuari creat correctament!")

L'usuari ja existeix: Toni Molist


##### **CONSULTA SIMPLE (SELECT)**

In [5]:
#SELECT * FROM USER
with Session() as session: 
    # 1. Fem la consulta per obtenir tots els usuaris
    stmt = select(User)
    # 2. Executem la consulta i obtenim els resultats
    users = session.execute(stmt)
    # 3. Iterem sobre els resultats i imprimim la informació de cada usuari
    for user in users:
        print(f"User: {user.User.name} {user.User.surname}, email: {user.User.email}")

User: Toni Molist, email: tonimolist@gmail.com
User: Joan Prova, email: test@walletly.com


##### **ORDENACIÓ (ORDER BY)**

In [6]:
with Session() as session:
    # 1. Fem la consulta per obtenir tots els usuaris ordenats per nom descendent
    stmt = select(User).order_by(User.name.desc()) 
    # 2. Executem la consulta i obtenim els resultats
    users = session.execute(stmt).all()
    # 3. Iterem sobre els resultats i imprimim la informació de cada usuari
    for user in users:
        print(f"User: {user.User.name} {user.User.surname}, email: {user.User.email}")

User: Toni Molist, email: tonimolist@gmail.com
User: Joan Prova, email: test@walletly.com


##### **FILTRES AVANÇATS**

In [7]:
with Session() as session:
    # 1. Fem la consulta per obtenir els usuaris que tinguin una T al nom o una M al cognom, ordenats per nom ascendent
    stmt = (
        select(User)
        .where(
            and_(
                or_(
                    User.name.ilike("%T%"),
                    User.surname.ilike("%M%")
                )
            )
        ) # Aquí tanquem el .where()
        .order_by(User.name) # L'ordre va FORA de les condicions
    )
    # 2. Executem la consulta i obtenim els resultats
    users = session.execute(stmt).all()
    # 3. Iterem sobre els resultats i imprimim la informació de cada usuari
    for user in users:
        print(f"User: {user.User.name} {user.User.surname}, email: {user.User.email}")

User: Toni Molist, email: tonimolist@gmail.com


##### **BUSCAR PER ID (ONE RESULT)**

In [8]:
with Session() as session:
    # 1. Fem la consulta per obtenir l'usuari amb id_user = 1
    stmt = select(User).where(User.id_user == 1)
    # 2. Executem la consulta i obtenim el resultat
    user = session.scalar(stmt)
    # 3. Imprimim la informació de l'usuari (si existeix)
    if user:
        print(f"User: {user.name} {user.surname}, email: {user.email}")
    

User: Toni Molist, email: tonimolist@gmail.com


##### **NAVEGACIÓ DE RELACIONS: USUARI -> PRESSUPOSTOS -> TRANSACCIONS  (GET BY)**

In [9]:
with Session() as session: 
    # 1. Fem la consulta per obtenir l'usuari amb id_user = 1
    user = session.get(User, 1)
    # 2. Imprimim la informació de l'usuari (si existeix)
    if user:
        # CORREGIT: 'user' ja és l'objecte, no cal posar '.User' entremig
        print(f"User: {user.name} {user.surname}, email: {user.email}")

        print("\nPressupostos i Transaccions:")
        for b in user.budget:
            # CORREGIT: 'description' i 'total_limit' són els camps de la teva taula Budget
            print(f" Budget: {b.description}, Límit: {b.total_limit}")
            
            # OPCIONAL: Si vols veure els imports reals (amount), has d'iterar les transaccions
            for t in b.transactions:
                print(f"   - Transacció: {t.description}, Import: {t.amount}")

User: Toni Molist, email: tonimolist@gmail.com

Pressupostos i Transaccions:
 Budget: Pressupost Abril inicial, Límit: 1500.00


##### **CONSULTA D'USUARI I CÀRREGA DE PRESSUPOSTOS (1:N) JOIN**

In [10]:
with Session() as session:
    # 1. Has de fer servir select(User) dins de scalars
    # Fem servir .first() o .one() per agafar un sol usuari de la llista
    user = session.scalars(select(User)).first() 

    # 2. Imprimim la informació de l'usuari (si existeix) i els seus pressupostos
    if user:
        print(f"User: {user.name} {user.surname}, email: {user.email}")

        print("\nFinancial Profile / Budgets:")

        # 3. Camps correctes: .description i .total_limit
        for b in user.budget:
            print(f"  Budget: {b.description}, limit: {b.total_limit}")

User: Toni Molist, email: tonimolist@gmail.com

Financial Profile / Budgets:
  Budget: Pressupost Abril inicial, limit: 1500.00


##### **RESUM ESTADISTIC: RECOMPTE DE PRESSUPOSTOS PER USUARI (GROUP BY)**

In [11]:
with Session() as session:
    # 1. Fem la consulta per obtenir el nombre de pressupostos per cada usuari, incloent els usuaris que no tinguin cap pressupost
    stmt = (
        select(User.name, User.surname, func.count(Budget.id_budget).label("num_budgets"))
        .outerjoin(Budget, Budget.id_user == User.id_user)
        .group_by(User.id_user)
    )
    # 2. Executem la consulta i obtenim els resultats
    rows = session.execute(stmt)
    # 3. Iterem sobre els resultats i imprimim el nombre de pressupostos per cada usuari
    for name, surname, num_budgets in rows:
        print(f"User: {name} {surname}, Number of Budgets: {num_budgets}")

User: Toni Molist, Number of Budgets: 1
User: Joan Prova, Number of Budgets: 0
